# 08 — Brain states × mapas de referência

Interpreta uma definição de cluster (um conjunto de brain states) em termos neuroanatômicos: a ativação média de cada estado no córtex e no subcórtex, e a correlação com mapas **moleculares** (PET, neuromaps), **comportamentais** (taxonomia BrainMap, perfis do Neurosynth calculados com o decodificador BrainMap do NiMARE nas parcelas do próprio atlas) e **canônicos** (Yeo 7 e as nossas 14 redes).

Duas etapas, dois módulos do pacote:

1. `fmri_decomposition.unsupervised_model.contextualization.correlations.compute(...)` → **tabelas** (`state_means.csv`, `correlations.csv`, `provenance.json`) em `outputs/modelling/version=<v>/unsupervised_model/contextualization/<estados>/<atlas>/`
2. `fmri_decomposition.figures.states.draw(...)` → **figuras** a partir dessas tabelas (visão geral + um card por estado)

Mapas de referência (~285 MB): ficam em `$FMRIDECOMP_REFMAPS`, senão em `<repo>/reference_atlases/` (no `.gitignore`). Para baixar: `fmri_decomposition/unsupervised_model/contextualization/fetch_reference_maps.sh`. Detalhes e limitações: `fmri_decomposition/unsupervised_model/contextualization/README.md`.

In [ ]:
import sys
from pathlib import Path

# THE PACKAGE, which is not pip-installed inside the analysis image. A Jupyter
# kernel starts with sys.path[0] = the notebook's own directory, so without
# this `import fmri_decomposition` fails from notebooks/.
#
# The REPO ROOT and nothing else. `notebooks/` used to be on this list too,
# back when the table and figure modules lived beside the .ipynb files; that
# part is gone and must not come back -- a directory on sys.path can shadow a
# real package, which is how an ImportError turns into the wrong module.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Image, display

from fmri_decomposition.unsupervised_model.contextualization import correlations as sc
from fmri_decomposition.figures import states as sf

In [ ]:
# Os perfis comportamentais usam o decodificador BrainMap do NiMARE (>= 0.22; o 0.4 tinha o
# zForward errado). No contêiner atual ele vem de uma pasta externa: APPTAINERENV_PYTHONPATH.
import nimare, nilearn
print('nimare', nimare.__version__, '| nilearn', nilearn.__version__)
assert tuple(int(x) for x in nimare.__version__.split('.')[:2]) >= (0, 22), 'nimare >= 0.22 needed'


## Parâmetros

**Definição de cluster** — `STATES`:
- `"simulate"`: seis estados simulados com resposta conhecida (para testar);
- uma coluna de estados do stage 4b, ex. `"HMM2_pca14_10"` (precisa de `OUTPUT_ROOT` e `WINDOW_S`);
- ou o caminho de um CSV escrito por `fmri-decomp state-means --csv`.

**Atlas** — `ATLASES`: onde os estados foram ajustados (`harvardoxford`, `networks_nodes`, `networks`, `yeo7`). Uma coluna de estados existe num atlas só; liste os atlas onde ela existe.

**O que mostrar** — `FAMILIES` (painéis não comportamentais) e `TAXONOMY`: `"brainmap"` (categorias do BrainMap, Fox et al. 2005, traduzidas em termos do Neurosynth) ou `"beam2021"` (6 domínios data-driven de Beam et al. 2021: Memory, Reward, Cognition, Vision, Manipulation, Language). `BN_LEVELS`: profundidade — `"domain"` (os domínios), `"subcategory"` (BrainMap: ex. Vision · Motion; Beam: os termos de cada domínio), ou os dois.

**Estatística** — `N_NULLS` substitutos por mapa (`NULL="moran"`, segundos; `"burt"`, brainsmash, horas). Atlas com < 30 parcelas (`networks`, `yeo7`) ficam sem p-valor.

**Figura** — uma linha por estado: o cérebro em 2 × 3 (lateral E · lateral D · axial / medial E · medial D · coronal) e, ao lado, um gráfico de barras por família em `PANELS` (as `TOP` maiores |r|; categorias "Other" do BrainMap ficam de fora). As barras são a média de r entre sujeitos **± desvio padrão entre sujeitos** quando os estados vêm de uma coluna de estados ou da simulação (cada sujeito tem o seu mapa de estado); com um CSV de `state-means` só existe o mapa do grupo e as barras saem sem erro.

In [ ]:
STATES      = "HMM2_umap3_8"        # o conjunto do manuscrito | "simulate" | "path/to/state_means.csv"
WINDOW_S    = 30                    # 30-s dFC (manuscrito); -1 = per-TR (activation)
COHORTS     = "train"               # "train" = só as coortes de treino do decompose | lista | None = todas
OUTPUT_ROOT = None                  # None = $FMRIDECOMP_OUTPUTS ou o output_root dos config/*.yaml

ATLASES     = ["harvardoxford"]     # + "networks_nodes", "networks", "yeo7"
FAMILIES    = ["molecular", "yeo7", "networks14"]
BN_LEVELS   = ["domain", "subcategory"]     # profundidade da taxonomia comportamental
BEHAVIOURAL = "neurosynth_lancaster_weighted"  # Neurosynth, estatística de Lancaster: "..._weighted" (focos com peso 1/n por estudo) | "..._reverse" (inferência reversa) | "neurosynth_lancaster" (focos) | "neurosynth_lancaster_studies" (estudos) | "brainmap" / "brainmap_reverse" (base BrainMap via Mango) | "brainmap_all" | "neurosynth" (zForward antigo) | None = simulado
TAXONOMY    = "brainmap"            # "brainmap" (Fox 2005, com sinônimos) | "brainmap_pure" (só o nome da categoria, sem sinônimos) | "beam2021" (Beam et al. 2021)

PANELS      = ["yeo7", "behavioural"]   # gráficos ao lado de cada cérebro, nessa ordem
TOP         = 7                         # barras por gráfico (Yeo 7 mostra sempre as 7)
POSITIVE_ONLY = ["behavioural"]          # gráficos só com r positivo; [] = positivos e negativos

N_NULLS     = 10_000
NULL        = "moran"
RESULTS     = None       # None = <outputs>/modelling/version=<v>/
                         #        unsupervised_model/contextualization/

## 1. Tabelas de correlação

In [ ]:
dirs = sc.compute(
    STATES, ATLASES,
    families=FAMILIES + [sc.BN_LEVELS[b] for b in BN_LEVELS],
    out=RESULTS, output_root=OUTPUT_ROOT, window_s=WINDOW_S, cohorts=COHORTS,
    behavioural_csv=BEHAVIOURAL, n_nulls=N_NULLS, null=NULL, taxonomy=TAXONOMY,
)
run_dir = next(iter(dirs.values())).parent
run_dir

In [ ]:
import pandas as pd
cor = pd.concat({a: pd.read_csv(d / "correlations.csv") for a, d in dirs.items()}, names=["atlas"])
cor.sort_values("p_null").groupby(["atlas", "family"]).head(3)

## 2. Figuras

In [ ]:
paths = sf.draw(run_dir, atlases=ATLASES, families=FAMILIES, bn_levels=BN_LEVELS,
                panels=PANELS, top=TOP, positive_only=POSITIVE_ONLY)
for p in paths:
    display(Image(filename=str(p)))

## 3. Matriz entre mapas de referência

Correlação espacial, entre as parcelas do atlas escolhido, entre cada mapa de uma família (linhas) e cada mapa de outra (colunas) — por padrão as 7 redes de Yeo × as 38 subcategorias do BrainMap (perfis do Neurosynth, zForward do decodificador BrainMap do NiMARE, contados nas parcelas do atlas; sem as "Other"). Não depende dos estados. Asteriscos: `*` q<.05, `**` q<.01, contra mapas substitutos que preservam a autocorrelação espacial, FDR sobre a matriz inteira. Famílias: `molecular`, `behavioural`, `behavioural_domain`, `yeo7`, `networks14`.

In [ ]:
MATRIX_ATLAS = "harvardoxford"     # "harvardoxford" | "networks_nodes" | "networks" | "yeo7"
MATRIX_ROWS  = "yeo7"
MATRIX_COLS  = "behavioural"
MATRIX_CORTICAL_ONLY = None    # None = só parcelas corticais quando Yeo 7 é um dos lados (Yeo é cortical) | True | False

png = sf.reference_matrix(MATRIX_ATLAS, MATRIX_ROWS, MATRIX_COLS, taxonomy=TAXONOMY,
                          behavioural_csv=BEHAVIOURAL, n_nulls=N_NULLS, null=NULL,
                          cortical_only=MATRIX_CORTICAL_ONLY)
display(Image(filename=str(png)))

## 4. Outras visualizações (opcional)

Descomente para gerar. Usam **todas as famílias** de `FAMILIES` + `BN_LEVELS` (moleculares, comportamentais, Yeo 7, nossas 14 redes), não só `PANELS`.

In [ ]:
# Matriz: uma linha por estado (cérebro) e uma faixa de calor por família, com todas as famílias
# sf.draw(run_dir, atlases=ATLASES, families=FAMILIES, bn_levels=BN_LEVELS, overview=True)
# for atlas in ATLASES:
#     display(Image(filename=str(dirs[atlas] / "state_maps_overview.png")))

In [ ]:
# Cards: um por estado, com barras de todas as famílias
# sf.draw(run_dir, atlases=ATLASES, families=FAMILIES, bn_levels=BN_LEVELS, cards=True)
# for atlas in ATLASES:
#     for card in sorted(dirs[atlas].glob("state_card_S*.png"),
#                        key=lambda p: int(p.stem.rsplit("S", 1)[1])):
#         print(atlas, card.stem)
#         display(Image(filename=str(card)))

## 5. Nomes dos estados

Regra (Métodos): **rede de Yeo dominante + domínio BrainMap dominante do polo positivo** — para cada estado, a rede com maior r positivo entre as 7 de Yeo e o domínio (Action, Cognition, Emotion, Interoception, Perception) com maior r positivo. Marcamos com `*` quando a associação passa q < .05 contra o nulo espacial. A tabela é uma proposta: confira com as figuras antes de usar os nomes no texto.


In [ ]:
NAME_ATLAS = ATLASES[0]
c = pd.read_csv(dirs[NAME_ATLAS] / "correlations.csv")

def top_positive(fam):
    t = c[(c["family"] == fam) & (c["r"] > 0)].sort_values("r", ascending=False)
    return t.groupby("state").head(1).set_index("state")

net, dom = top_positive("yeo7"), top_positive("behavioural_domain")
names = pd.DataFrame(index=sorted(c["state"].unique()))
names["network"] = net["map"]
names["r_network"] = net["r"].round(2)
names["q_network"] = net["q_fdr"]
names["domain"] = dom["map"]
names["r_domain"] = dom["r"].round(2)
names["q_domain"] = dom["q_fdr"]
names["proposed_name"] = [
    f"{n}{'*' if pd.notna(qn) and qn < .05 else ''}–{d}{'*' if pd.notna(qd) and qd < .05 else ''}"
    for n, d, qn, qd in zip(names["network"], names["domain"], names["q_network"], names["q_domain"])]
names
